# Receptor refinement geometry — Hallem--Carlson (fly (Drosophila))

Insect analogue of `receptor_refinement_2d_M2OR.ipynb`. It takes the Hallem--Carlson signed-graph
run (transductive `rand` split, five folds) and shows, as 2D PCA scatters, the receptor
cloud **before** message passing (raw mean-pooled ESM-1b) and the **five refined** clouds
`z[PROT]` (one per fold).

**No family taxonomy here.** Insect ORs have no vertebrate-style `OR<n>` families, so —
instead of a sequence taxonomy — receptors are grouped **by data, two ways**: *functional*
clusters (agglomerative on the binding-response profile over all {110} odorants) and
*structural* clusters (agglomerative on raw ESM). The clouds are coloured by the functional
group. The mechanism to look for is the same as M2OR, read through these two axes: the
graph should **trade structural geometry for functional geometry** — refined clouds sharpen
the functional grouping (silhouette/purity up) while loosening the structural one — and
§8 checks alignment with continuous functional (binding-profile) proximity directly. Since
the Hallem--Carlson matrix is complete (every receptor tested on every odorant), the functional
profile is dense and the proximity metric is exact.


In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)   # orbind resolves its data/ paths relative to CWD

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from orbind.regimes_ofm import ofm_pairs, ofm_indices
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict,
    _train_one, MOL, PROT,
)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# --- which run's models to visualise -------------------------------------
# Hallem--Carlson transductive (rand) signed-graph run: 5 folds, each a trained _SignedSage
# (regression). Checkpoints under RUN_DIR/checkpoints/repeat_{fold}/gnn_{CLS_NAME}_model0.pt
# are loaded if present, else the graph is reproduced with the same config (it is tiny).
DS       = "hc"
RUN_DIR  = ROOT / "results/ensemble_logs/hc-rand-molcross-fixed/hc_rand_gnn_nodeGIN_concatGIN"
CLS_NAME = "cls"
FAMILY   = "rand"          # transductive: every receptor is seen, so refinement is defined
FOLDS    = [1, 2, 3, 4, 5]

# graph config = the hc molcross `cls=gnn_signed` source, verbatim
# (complete insect matrix -> q=0 / criterion=coverage; edge_threshold 0 = response above pool mean)
PROT_NPZ = DATA / "embeddings/proteins/esm1b_650m_mean_hc.npz"
MOL_NPZ  = DATA / "embeddings/molecules/gin_supervised_contextpred_hc.npz"
GRAPH_KW = dict(q=0.0, criterion="coverage", edge_threshold=0.0, k_mode="coverage_quantile",
                emit="prot", n_models=1, hidden=256, task="regression")

N_GROUPS = 5   # functional / structural clusters (no OR taxonomy for insects)
KNN      = 5

## 1 · Load / reproduce the five models and pull the embeddings

Per fold we rebuild the transductive universe and message-passing graph, load (or retrain)
the encoder, and read off the **initial** raw-ESM features (shared across folds) and the
**refined** receptor vectors `z[PROT]`.


In [ ]:
pairs = ofm_pairs(DS)
ext = GnnSignedExtractor(name=CLS_NAME, protein_path=str(PROT_NPZ),
                         molecule_path=str(MOL_NPZ), **GRAPH_KW)

initial, refined = None, {}
for fold in FOLDS:
    tr, va, te = ofm_indices(DS, FAMILY, fold)
    tr, va, te = (i[ext.covered(pairs, i)] for i in (tr, va, te))
    all_idx = np.concatenate([tr, va, te])
    mol_to_i, prot_to_i, x_mol, x_prot = _build_universe(pairs, all_idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(pairs, tr, mol_to_i, prot_to_i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pos_eidx, neg_eidx = _edge_index_dict(pos, neg)

    tdf = pairs.iloc[tr]
    mi = tdf["inchikey"].map(mol_to_i).to_numpy()
    pi = tdf["receptor"].map(prot_to_i).to_numpy()
    y  = tdf["label"].to_numpy(dtype=np.float32)

    ckpt = RUN_DIR / "checkpoints" / f"repeat_{fold}" / f"gnn_{CLS_NAME}_model0.pt"
    z_mol, z_prot, model = _train_one(
        ext._build_model, x_mol, x_prot, pos_eidx, neg_eidx, mi, pi, y,
        ext._hp(fold), DEVICE, checkpoint_path=(ckpt if ckpt.exists() else None))

    refined[fold] = {r: z_prot[i] for r, i in prot_to_i.items()}
    if initial is None:
        initial = {r: x_prot[i].numpy() for r, i in prot_to_i.items()}
    print(f"fold {fold}: {'loaded' if ckpt.exists() else 'reproduced':10s} "
          f"| {len(prot_to_i)} receptors | {len(pos[0])}+/{len(neg[0])}- MP edges "
          f"| refined dim {z_prot.shape[1]}")

REC          = sorted(set(initial).intersection(*[set(refined[f]) for f in FOLDS]))
init_mat     = np.stack([initial[r] for r in REC])
refined_mats = {f: np.stack([refined[f][r] for r in REC]) for f in FOLDS}
print(f"\naligned on {len(REC)} receptors | init {init_mat.shape} | refined {refined_mats[FOLDS[0]].shape}")

## 2 · Two data-driven groupings (no taxonomy) + colour

Every receptor is tested on all odorants, so its **binding-response profile** (a dense
continuous vector) is a clean functional fingerprint. We derive two labelings by
agglomerative clustering into 5 groups each: **functional** (on the profile) and
**structural** (on raw ESM). The clouds are coloured by the *functional* group. There is no
`other` bucket — the Hallem--Carlson receptor set is small and fully curated, so nothing is dropped.


In [ ]:
from sklearn.cluster import AgglomerativeClustering

# dense continuous binding-response profile: receptor x odorant
mols  = sorted(pairs["inchikey"].unique())
r_idx = {r: i for i, r in enumerate(REC)}
m_idx = {m: j for j, m in enumerate(mols)}
g = pairs[pairs["receptor"].isin(set(REC))].groupby(["receptor", "inchikey"], as_index=False)["label"].mean()
P = np.full((len(REC), len(mols)), np.nan, np.float32)
P[g["receptor"].map(r_idx).to_numpy(), g["inchikey"].map(m_idx).to_numpy()] = g["label"].to_numpy()
P = np.nan_to_num(P, nan=0.0)
print(f"binding profile: {P.shape[0]} receptors x {P.shape[1]} odorants (dense)")

func_grp = AgglomerativeClustering(n_clusters=N_GROUPS).fit_predict(P)            # functional
seq_grp  = AgglomerativeClustering(n_clusters=N_GROUPS).fit_predict(init_mat)     # structural (raw ESM)

# colour by functional group
fam = np.array([f"F{c}" for c in func_grp])
fam_levels = sorted(set(fam))
cmap = plt.get_cmap("tab10", len(fam_levels))
color_map = {f: cmap(i) for i, f in enumerate(fam_levels)}
print("functional group sizes:", np.bincount(func_grp).tolist(),
      "| structural group sizes:", np.bincount(seq_grp).tolist())

## 3 · Initial cloud — raw ESM-1b

Mean-pooled ESM-1b, PCA to 2D, coloured by functional group. Raw sequence embedding should
show little functional organisation.


In [ ]:
from sklearn.decomposition import PCA

def to2d(M):
    return PCA(n_components=2, random_state=0).fit_transform(np.asarray(M, dtype=np.float64))

def scatter2d(ax, coords, title):
    for f in fam_levels:
        m = fam == f
        if not m.any():
            continue
        ax.scatter(coords[m, 0], coords[m, 1], s=40, alpha=0.85,
                   color=color_map[f], label=f, edgecolors="none")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.tick_params(labelsize=7)

fig, ax = plt.subplots(figsize=(7, 6))
scatter2d(ax, to2d(init_mat), f"Initial receptors — raw ESM-1b ({init_mat.shape[1]}d) -> PCA-2")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8,
          frameon=False, title="functional group")
fig.tight_layout()
plt.show()

## 4 · Refined clouds — one per fold (5 models)

Graph-refined receptor vectors. Points that share a functional group should draw together
relative to the raw cloud.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, f in zip(axes.ravel(), FOLDS):
    scatter2d(ax, to2d(refined_mats[f]), f"Refined — fold {f} ({refined_mats[f].shape[1]}d) -> PCA-2")
axes.ravel()[-1].axis("off")
handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center", bbox_to_anchor=(0.83, 0.25),
           fontsize=9, frameon=False, title="functional group")
fig.suptitle("Graph-refined receptor clouds (PCA-2), one per fold", y=1.0, fontsize=12)
fig.tight_layout()
plt.show()

## 5 · Structural vs functional grouping — the trade

The insect analogue of M2OR's *family vs function*. We score both labelings — **structural**
(ESM clusters) and **functional** (profile clusters) — on the full-dimensional vectors, raw
ESM vs each refined model, with silhouette and kNN purity. The graph should **loosen the
structural grouping and tighten the functional one**: refinement rebuilds receptor geometry
around binding, not sequence.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

def knn_purity(X, labels, k=KNN):
    k = min(k, len(X) - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    idx = nn.kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([np.mean(labels[nbrs] == labels[i]) for i, nbrs in enumerate(idx)]))

def scores(X, labels):
    return silhouette_score(X, labels), knn_purity(X, labels)

rows = []
for name, X in [("raw ESM", init_mat)] + [(f"refined f{f}", refined_mats[f]) for f in FOLDS]:
    s_sil, s_pur = scores(X, seq_grp)
    f_sil, f_pur = scores(X, func_grp)
    rows.append((name, s_sil, s_pur, f_sil, f_pur))
geo = pd.DataFrame(rows, columns=["embedding", "struct_sil", "struct_purity",
                                  "func_sil", "func_purity"])
r = geo.iloc[1:]
print("structural grouping (should DROP):  raw sil %.3f purity %.3f -> refined sil %.3f purity %.3f"
      % (geo.iloc[0].struct_sil, geo.iloc[0].struct_purity, r.struct_sil.mean(), r.struct_purity.mean()))
print("functional grouping (should RISE):  raw sil %.3f purity %.3f -> refined sil %.3f purity %.3f"
      % (geo.iloc[0].func_sil, geo.iloc[0].func_purity, r.func_sil.mean(), r.func_purity.mean()))
display(geo.style.format({c: "{:+.3f}" if "sil" in c else "{:.3f}"
                          for c in geo.columns if c != "embedding"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8.5, 4))
x = np.arange(len(geo)); w = 0.4
ax.bar(x - w/2, geo.struct_purity, w, label="structural (ESM) group", color="#6f63b6")
ax.bar(x + w/2, geo.func_purity,  w, label="functional (profile) group", color="#e4794c")
ax.set_xticks(x); ax.set_xticklabels(geo.embedding, rotation=25, ha="right", fontsize=8)
ax.set_ylabel(f"kNN purity (k={KNN})")
ax.set_title("Refinement trades structural grouping for functional grouping")
ax.legend(frameon=False); fig.tight_layout(); plt.show()

## 5b · Biological label: specialist vs generalist (tuning breadth)

Insect ORs are classically split into narrowly-tuned **specialists** (a few strong ligands
-- e.g. *Drosophila* Or56a/geosmin, Or67d/cVA pheromone, mosquito receptors for single
human-odour cues) and broadly-tuned **generalists** (Hallem & Carlson 2006, the source of
the Hallem--Carlson response matrix). We read tuning breadth straight off the profile (number of
odorants with a strong z>1 response) and tercile it into narrow / medium / broad, then ask
whether refinement organises the receptor geometry by this functional axis.


In [ ]:
breadth = (P > 1.0).sum(1)                       # # odorants with strong (z>1) response
q1, q2  = np.quantile(breadth, [1/3, 2/3])
tune    = np.where(breadth <= q1, "narrow", np.where(breadth >= q2, "broad", "medium"))
name_col = next((c for c in ("RECEPTOR", "Protein", "receptor_id") if c in pairs.columns), None)
if name_col:
    nm = pairs.drop_duplicates("receptor").set_index("receptor")[name_col].astype(str).to_dict()
    print("narrowest-tuned (specialists):", [nm.get(REC[i], "?") for i in np.argsort(breadth)[:6]])
print("tuning breadth  narrow/medium/broad:", [int((tune == t).sum()) for t in ["narrow", "medium", "broad"]])

tc = {"narrow": "#d1495b", "medium": "#c0c0c0", "broad": "#2e86ab"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (Mx, ttl) in zip(axes, [(init_mat, "raw ESM"), (refined_mats[FOLDS[0]], f"refined fold {FOLDS[0]}")]):
    co = to2d(Mx)
    for t in ["medium", "narrow", "broad"]:
        m = tune == t
        if m.any():
            ax.scatter(co[m, 0], co[m, 1], s=40, alpha=0.85, color=tc[t], label=t, edgecolors="none")
    ax.set_title(f"tuning breadth - {ttl}", fontsize=10); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[0].legend(frameon=False, fontsize=9, title="tuning")
fig.tight_layout(); plt.show()

rows = [("raw ESM", silhouette_score(init_mat, tune), knn_purity(init_mat, tune))]
rows += [(f"refined f{f}", silhouette_score(refined_mats[f], tune), knn_purity(refined_mats[f], tune)) for f in FOLDS]
tt = pd.DataFrame(rows, columns=["embedding", "tuning_sil", "tuning_purity"])
rr = tt.iloc[1:]
print(f"tuning breadth: raw sil {tt.iloc[0,1]:+.3f} purity {tt.iloc[0,2]:.3f} -> "
      f"refined sil {rr.iloc[:,1].mean():+.3f} purity {rr.iloc[:,2].mean():.3f}")
display(tt.style.format({"tuning_sil": "{:+.3f}", "tuning_purity": "{:.3f}"}).hide(axis="index"))

## 6 · Do the five models agree on the geometry?

Linear **CKA** between every pair of spaces: refined-vs-refined high = a reproducible
geometry across folds; init-vs-refined low = a real move away from raw ESM.


In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    return float((np.linalg.norm(Y.T @ X) ** 2) /
                 (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)))

mats = {"init": init_mat, **{f"f{f}": refined_mats[f] for f in FOLDS}}
keys = list(mats)
C = np.array([[linear_cka(mats[a], mats[b]) for b in keys] for a in keys])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(C, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys)
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys)
for i in range(len(keys)):
    for j in range(len(keys)):
        ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center",
                color="w" if C[i, j] < 0.6 else "k", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Linear CKA — raw init vs 5 refined models")
fig.tight_layout(); plt.show()

rr = C[1:, 1:][np.triu_indices(len(FOLDS), 1)]
print(f"refined<->refined CKA: {rr.mean():.3f} +/- {rr.std():.3f}  |  "
      f"init<->refined CKA: {C[0, 1:].mean():.3f} +/- {C[0, 1:].std():.3f}")

## 7 · Embedding geometry vs functional (binding-profile) proximity

The encoder-independent test. Functional similarity of two receptors = **correlation of
their response profiles over all odorants** (the matrix is complete, so every pair is
directly comparable). We compare the top-neighbour overlap and the full-matrix Spearman
between functional proximity and raw-ESM vs each refined space. Refined should align better.


In [ ]:
from scipy.stats import spearmanr

# functional similarity = Pearson correlation of profiles (mean-centred cosine)
Pc = P - P.mean(1, keepdims=True)
Pn = Pc / (np.linalg.norm(Pc, axis=1, keepdims=True) + 1e-9)
F  = Pn @ Pn.T

def emb_sim(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T

def topk_idx(S, k):
    S = S.copy(); np.fill_diagonal(S, -np.inf)
    return np.argpartition(-S, k, axis=1)[:, :k]

def overlap_at_k(Sfun, Semb, k=KNN):
    A, Bk = topk_idx(Sfun, k), topk_idx(Semb, k)
    return float(np.mean([len(set(A[i]) & set(Bk[i])) / k for i in range(len(A))]))

iu = np.triu_indices(len(REC), 1)
f_flat = F[iu]

def align(X):
    S = emb_sim(X)
    return overlap_at_k(F, S), spearmanr(f_flat, S[iu]).correlation

rows = [("raw ESM", *align(init_mat))] + [(f"refined f{f}", *align(refined_mats[f])) for f in FOLDS]
fa = pd.DataFrame(rows, columns=[f"embedding", f"overlap@{KNN}_functional", "spearman_functional"])
r = fa.iloc[1:]
print(f"raw ESM : overlap@{KNN} {fa.iloc[0,1]:.3f} | Spearman {fa.iloc[0,2]:+.3f}")
print(f"refined : overlap@{KNN} {r.iloc[:,1].mean():.3f} +/- {r.iloc[:,1].std():.3f} | "
      f"Spearman {r.iloc[:,2].mean():+.3f} +/- {r.iloc[:,2].std():.3f} (mean over 5 folds)")
display(fa.style.format({fa.columns[1]: "{:.3f}", "spearman_functional": "{:+.3f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(fa)), fa.iloc[:, 1], color=["#888888"] + ["#e4794c"] * len(FOLDS))
ax.axhline(fa.iloc[0, 1], ls=":", c="#888888", label="raw ESM")
ax.set_xticks(range(len(fa))); ax.set_xticklabels(fa.embedding, rotation=25, ha="right", fontsize=8)
ax.set_ylabel(f"overlap@{KNN} with functional neighbours")
ax.set_title("Do embeddings agree with binding-profile (functional) neighbours?")
ax.legend(frameon=False); fig.tight_layout(); plt.show()

**Takeaway (Hallem--Carlson).** With no OR taxonomy to lean on, the two data-driven groupings make
the mechanism legible: refinement **loosens** the structural (ESM) grouping and **tightens**
the functional (binding-profile) one (§5), and — encoder-independently — the refined spaces
sit closer to continuous functional proximity than raw ESM (§7), reproducibly across folds
(§6). Same receptor-side story as M2OR, on an insect dataset with a completely different
sequence space: the graph rebuilds receptor geometry from binding data into a **functional**
one.
